# Madrid por distritos — datos REALES del INE (Atlas de Renta 2023)

Análisis con datos **oficiales** del INE (Atlas de Distribución de Renta de los Hogares, 2023), a nivel de los **21 distritos** de la ciudad de Madrid.

**Requisito:** haber descargado los datos antes con `python src/extract.py` (deja 4 CSV en `data/raw/`). Este notebook los lee con `src/load_ine.py`.

> Nota: la descarga es un paso aparte (se hace una vez). El notebook solo **analiza**.


## 1. Cargar los datos reales


In [ ]:
import sys, os, importlib
SRC = os.path.abspath('../src')
if SRC not in sys.path: sys.path.insert(0, SRC)
import load_ine; importlib.reload(load_ine)
import pandas as pd, matplotlib.pyplot as plt

mad = load_ine.construir_madrid()   # lee data/raw, filtra Madrid, une las tablas
print(mad.shape[0], 'distritos ·', mad.shape[1], 'columnas')
mad[['cod_distrito','nombre_distrito','Renta neta media por hogar','Índice de Gini']].head(21)


## 2. Ranking de renta por distrito

Renta neta media por hogar (€/año). El noroeste arriba, el sur abajo.


In [ ]:
col = 'Renta neta media por hogar'
d = mad.dropna(subset=[col]).sort_values(col)
colors = ['#de2d26' if v < d[col].median() else '#2c7fb8' for v in d[col]]
fig, ax = plt.subplots(figsize=(9,8))
ax.barh(d['nombre_distrito'], d[col], color=colors)
ax.set_xlabel(col + ' (€/año)')
ax.set_title('Madrid · renta REAL por distrito (INE 2023)')
plt.tight_layout(); plt.show()

print('Más rico: ', d.iloc[-1]['nombre_distrito'], int(d.iloc[-1][col]), '€')
print('Más pobre:', d.iloc[0]['nombre_distrito'], int(d.iloc[0][col]), '€')
print('Ratio rico/pobre:', round(d.iloc[-1][col]/d.iloc[0][col], 2), 'veces')


## 3. El hallazgo: ¿los distritos ricos son más desiguales?

Cruzamos la renta con el **Índice de Gini** (desigualdad *dentro* del distrito). El tamaño del punto = población; el color = % de renta que viene de salario.


In [ ]:
x, y = 'Renta neta media por hogar', 'Índice de Gini'
color = 'ingresos_Fuente de ingreso: salario'
size = 'demo_Población'
fig, ax = plt.subplots(figsize=(8.5,6.5))
sc = ax.scatter(mad[x], mad[y], s=mad[size]/2500, c=mad[color], cmap='RdYlGn')
for _, r in mad.iterrows():
    ax.annotate(r['nombre_distrito'], (r[x], r[y]), fontsize=6, alpha=0.75)
ax.set_xlabel(x + ' (€)'); ax.set_ylabel(y + ' (desigualdad)')
ax.set_title('Madrid · renta vs desigualdad (INE 2023)')
plt.colorbar(sc, label='% de renta que viene de salario')
plt.tight_layout(); plt.show()

print('Correlación renta ↔ Gini:', round(mad[x].corr(mad[y]), 2))
print('Correlación renta ↔ %salario:', round(mad[x].corr(mad[color]), 2))


**Lectura:** una correlación renta↔Gini positiva significa que los distritos con más renta son también los más desiguales por dentro. Y la correlación renta↔%salario negativa sugiere que en los distritos ricos una parte mayor de la renta NO viene del salario (rentas del capital, etc.).


## 4. Capa DuckDB — consultar los datos con SQL

Guardamos el dataset en una base de datos **DuckDB** (un solo fichero, sin servidor) y lo consultamos con SQL. Esto es 'tu base de datos' sin montar nada.


In [ ]:
import duckdb
con = duckdb.connect('../outputs/madrid.duckdb')
con.execute('CREATE OR REPLACE TABLE distritos AS SELECT * FROM mad')

# ejemplo de consulta SQL sobre los datos reales
q = '''
  SELECT nombre_distrito,
         "Renta neta media por hogar" AS renta_hogar,
         "Índice de Gini"            AS gini
  FROM distritos
  ORDER BY renta_hogar DESC
  LIMIT 5
'''
con.execute(q).df()


In [ ]:
# las consultas devuelven DataFrames de pandas, listos para graficar
pobres = con.execute('''
  SELECT nombre_distrito, "Renta neta media por hogar" AS renta
  FROM distritos ORDER BY renta ASC LIMIT 5
''').df()
con.close()
pobres


## 5. Conclusiones

- **Datos 100% oficiales** (INE Atlas de Renta 2023), reproducibles con `python src/extract.py`.
- Brecha real entre distritos: el más rico casi **duplica o más** la renta del más pobre.
- Los distritos de **mayor renta** tienden a ser los **más desiguales** internamente.

**Siguiente paso:** añadir la capa de **vivienda** (alquiler/compra por distrito) para calcular el *esfuerzo* real, y el **mapa** con la geometría de distritos del Ayto. de Madrid.

**Limitaciones:** la renta es por hogar/persona del Atlas (registros administrativos); el dato es por distrito, no por individuo.
